# Phase 14 — Expanded Semantic Representation + Cross-Dataset Improvement

**NetShield goal:** improve cross-dataset intrusion-detection performance, not only explain dataset shift.

This phase directly tests whether the current 11-feature semantic representation is too restrictive. It compares the same Random Forest pipeline using:
1. 11-feature semantic representation
2. Expanded semantic representation using only mathematically derived, direction/meaning-preserving features
3. Expanded semantic + CORAL
4. Expanded semantic + CORAL + class weighting

**Leakage rule:** CICIDS labels are never used for fitting preprocessing, CORAL, thresholds, feature selection, or pseudo-labels. CICIDS labels are used only for the final frozen evaluation.


## Why this phase matters

The current best cross-dataset result is about F1 = 0.339 with semantic features + CORAL + class weighting. The purpose here is to test whether information loss from the compact 11-feature representation is limiting transfer performance.

We do **not** add questionable raw-feature mappings. The expanded features below are deterministic transformations of the already verified common quantities, such as packet/byte fractions, bytes per packet, log-scale versions, and rates. This makes the representation richer without inventing unsupported cross-dataset semantics.


In [ ]:

from pathlib import Path
import gc
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    average_precision_score, confusion_matrix
)
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

# Robust project-root discovery
candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None
for root in candidates:
    if (root / "data" / "raw").exists() or (root / "outputs").exists():
        PROJECT_ROOT = root
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()

RAW_UNSW = PROJECT_ROOT / "data" / "raw" / "UNSW-NB15"
RAW_CICIDS = PROJECT_ROOT / "data" / "raw" / "CICIDS2017"

TABLE_DIR = PROJECT_ROOT / "outputs" / "expanded_semantic_transfer" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_UNSW:", RAW_UNSW)
print("RAW_CICIDS:", RAW_CICIDS)


In [ ]:

# -----------------------------
# UNSW-NB15 common representation
# -----------------------------
def load_unsw_common():
    all_csvs = list(RAW_UNSW.rglob("*.csv"))

    train_files = [p for p in all_csvs if "training" in p.name.lower()]
    test_files = [p for p in all_csvs if "testing" in p.name.lower()]

    if not train_files or not test_files:
        raise FileNotFoundError(
            f"Could not find UNSW training/testing CSVs under {RAW_UNSW}"
        )

    train_file = sorted(train_files)[0]
    test_file = sorted(test_files)[0]

    print("UNSW training:", train_file)
    print("UNSW testing :", test_file)

    tr = pd.read_csv(train_file, low_memory=False)
    te = pd.read_csv(test_file, low_memory=False)
    df = pd.concat([tr, te], ignore_index=True)

    y = pd.to_numeric(df["label"], errors="coerce").astype("Int8")

    dur = pd.to_numeric(df["dur"], errors="coerce")
    fp = pd.to_numeric(df["spkts"], errors="coerce")
    bp = pd.to_numeric(df["dpkts"], errors="coerce")
    fb = pd.to_numeric(df["sbytes"], errors="coerce")
    bb = pd.to_numeric(df["dbytes"], errors="coerce")

    total_p = fp + bp
    total_b = fb + bb
    safe_dur = dur.replace(0, np.nan)
    safe_total_p = total_p.replace(0, np.nan)

    x = pd.DataFrame(index=df.index)
    x["flow_duration_sec"] = dur
    x["fwd_packets"] = fp
    x["bwd_packets"] = bp
    x["fwd_bytes"] = fb
    x["bwd_bytes"] = bb
    x["total_packets"] = total_p
    x["total_bytes"] = total_b
    x["packet_rate"] = total_p / safe_dur
    x["fwd_packet_rate"] = fp / safe_dur
    x["bwd_packet_rate"] = bp / safe_dur
    x["avg_packet_size"] = total_b / safe_total_p

    return x, y


# -----------------------------
# CICIDS2017 common representation
# -----------------------------
def load_cicids_common():
    needed = [
        "Flow Duration",
        "Total Fwd Packets",
        "Total Backward Packets",
        "Total Length of Fwd Packets",
        "Total Length of Bwd Packets",
        "Flow Packets/s",
        "Fwd Packets/s",
        "Bwd Packets/s",
        "Average Packet Size",
        "Label",
    ]

    frames = []

    for p in sorted(RAW_CICIDS.rglob("*.csv")):
        h = pd.read_csv(p, nrows=0)
        lookup = {
            str(c).strip().replace("﻿", "").lower(): c
            for c in h.columns
        }

        sel = {
            c: lookup[c.lower()]
            for c in needed
            if c.lower() in lookup
        }

        if "Label" not in sel:
            print("Skipping:", p.name)
            continue

        chunks = []

        for chunk in pd.read_csv(
            p,
            usecols=list(sel.values()),
            chunksize=100_000,
            low_memory=True
        ):
            chunk = chunk.rename(
                columns={v: k for k, v in sel.items()}
            )

            for c in needed:
                if c != "Label" and c in chunk.columns:
                    chunk[c] = pd.to_numeric(
                        chunk[c], errors="coerce"
                    )

            chunk["Label"] = (
                chunk["Label"].astype(str).str.strip()
            )
            chunks.append(chunk)

        file_df = pd.concat(chunks, ignore_index=True)
        frames.append(file_df)

        print("Loaded:", p.name, len(file_df))

        del chunks, file_df
        gc.collect()

    if not frames:
        raise ValueError("No CICIDS CSV with Label found.")

    df = pd.concat(frames, ignore_index=True)

    x = pd.DataFrame(index=df.index)
    x["flow_duration_sec"] = df["Flow Duration"] / 1e6
    x["fwd_packets"] = df["Total Fwd Packets"]
    x["bwd_packets"] = df["Total Backward Packets"]
    x["fwd_bytes"] = df["Total Length of Fwd Packets"]
    x["bwd_bytes"] = df["Total Length of Bwd Packets"]
    x["total_packets"] = (
        x["fwd_packets"] + x["bwd_packets"]
    )
    x["total_bytes"] = (
        x["fwd_bytes"] + x["bwd_bytes"]
    )
    x["packet_rate"] = df["Flow Packets/s"]
    x["fwd_packet_rate"] = df["Fwd Packets/s"]
    x["bwd_packet_rate"] = df["Bwd Packets/s"]
    x["avg_packet_size"] = df["Average Packet Size"]

    y = (
        df["Label"]
        .astype(str)
        .str.strip()
        .str.upper()
        .ne("BENIGN")
        .astype("int8")
    )

    return x, y


X_unsw_11, y_unsw = load_unsw_common()
X_cic_11, y_cic = load_cicids_common()

print("UNSW:", X_unsw_11.shape)
print("CICIDS:", X_cic_11.shape)


## 1. Build the expanded semantic representation

These are derived only from the verified 11 semantic quantities. No target labels are involved.


In [ ]:

COMMON_11 = [
    "flow_duration_sec",
    "fwd_packets",
    "bwd_packets",
    "fwd_bytes",
    "bwd_bytes",
    "total_packets",
    "total_bytes",
    "packet_rate",
    "fwd_packet_rate",
    "bwd_packet_rate",
    "avg_packet_size",
]


def build_expanded_semantic(X):
    x = X[COMMON_11].copy()

    eps = 1e-9

    # Direction fractions
    x["fwd_packet_fraction"] = (
        x["fwd_packets"] /
        (x["total_packets"] + eps)
    )
    x["bwd_packet_fraction"] = (
        x["bwd_packets"] /
        (x["total_packets"] + eps)
    )

    x["fwd_byte_fraction"] = (
        x["fwd_bytes"] /
        (x["total_bytes"] + eps)
    )
    x["bwd_byte_fraction"] = (
        x["bwd_bytes"] /
        (x["total_bytes"] + eps)
    )

    # Directional ratios
    x["fwd_bwd_packet_ratio"] = (
        x["fwd_packets"] / (x["bwd_packets"] + eps)
    )
    x["fwd_bwd_byte_ratio"] = (
        x["fwd_bytes"] / (x["bwd_bytes"] + eps)
    )

    # Bytes per packet by direction
    x["fwd_bytes_per_packet"] = (
        x["fwd_bytes"] / (x["fwd_packets"] + eps)
    )
    x["bwd_bytes_per_packet"] = (
        x["bwd_bytes"] / (x["bwd_packets"] + eps)
    )

    # Overall bytes per packet
    x["bytes_per_packet"] = (
        x["total_bytes"] /
        (x["total_packets"] + eps)
    )

    # Byte rate derived from common quantities
    safe_duration = x["flow_duration_sec"].replace(0, np.nan)
    x["byte_rate"] = (
        x["total_bytes"] / safe_duration
    )

    # Directional byte rates
    x["fwd_byte_rate"] = (
        x["fwd_bytes"] / safe_duration
    )
    x["bwd_byte_rate"] = (
        x["bwd_bytes"] / safe_duration
    )

    # Log transforms reduce the effect of extreme scale differences.
    # Keep original features as well.
    for c in [
        "flow_duration_sec",
        "fwd_packets",
        "bwd_packets",
        "fwd_bytes",
        "bwd_bytes",
        "total_packets",
        "total_bytes",
        "packet_rate",
        "fwd_packet_rate",
        "bwd_packet_rate",
        "avg_packet_size",
        "byte_rate",
        "fwd_byte_rate",
        "bwd_byte_rate",
    ]:
        x[f"log1p_{c}"] = np.log1p(
            x[c].clip(lower=0)
        )

    return x.replace([np.inf, -np.inf], np.nan)


X_unsw_exp = build_expanded_semantic(X_unsw_11)
X_cic_exp = build_expanded_semantic(X_cic_11)

print("11-feature representation:", len(COMMON_11))
print("Expanded representation:", X_unsw_exp.shape[1])

print("\nExpanded features:")
print(list(X_unsw_exp.columns))


In [ ]:

# Basic invalid-value cleaning.
# We do NOT delete outliers and do NOT change legitimate zeros.
def clean_numeric(X):
    X = X.copy()
    X = X.replace([np.inf, -np.inf], np.nan)

    # Values outside float32 range are unusable numerical artifacts.
    f32_max = np.finfo(np.float32).max
    X = X.mask(X.abs() > f32_max)

    return X


X_unsw_11 = clean_numeric(X_unsw_11)
X_cic_11 = clean_numeric(X_cic_11)
X_unsw_exp = clean_numeric(X_unsw_exp)
X_cic_exp = clean_numeric(X_cic_exp)

print("Cleaning complete.")


## 2. Create leakage-safe source and target partitions

UNSW is split into source-fit and source-validation data. CICIDS is randomly partitioned without using its labels: one portion is used only as **unlabeled adaptation data**, and a separate portion is frozen for final evaluation.


In [ ]:

source_fit_idx, source_val_idx = train_test_split(
    np.arange(len(y_unsw)),
    test_size=0.20,
    stratify=y_unsw,
    random_state=RANDOM_STATE
)

# Target partition is deliberately NOT stratified by y_cic.
# Labels are not used to construct the adaptation set.
rng = np.random.default_rng(RANDOM_STATE)
target_order = rng.permutation(len(y_cic))

adaptation_n = min(300_000, len(target_order) // 2)
eval_n = min(300_000, len(target_order) - adaptation_n)

adapt_idx = target_order[:adaptation_n]
eval_idx = target_order[adaptation_n:adaptation_n + eval_n]

print("UNSW source fit:", len(source_fit_idx))
print("UNSW source validation:", len(source_val_idx))
print("CICIDS unlabeled adaptation:", len(adapt_idx))
print("CICIDS frozen evaluation:", len(eval_idx))

# Target labels are only inspected here to document the frozen evaluation.
print("\nFrozen CICIDS evaluation attack rate:",
      float(y_cic.iloc[eval_idx].mean()))


In [ ]:

def prepare_arrays(X_source, X_target_adapt, X_target_eval):
    imp = SimpleImputer(strategy="median")

    Xs_fit = imp.fit_transform(
        X_source.iloc[source_fit_idx]
    ).astype("float64")

    Xs_val = imp.transform(
        X_source.iloc[source_val_idx]
    ).astype("float64")

    Xt_adapt = imp.transform(
        X_target_adapt.iloc[adapt_idx]
    ).astype("float64")

    Xt_eval = imp.transform(
        X_target_eval.iloc[eval_idx]
    ).astype("float64")

    return Xs_fit, Xs_val, Xt_adapt, Xt_eval, imp


y_source_fit = y_unsw.iloc[source_fit_idx].to_numpy()
y_source_val = y_unsw.iloc[source_val_idx].to_numpy()
y_target_eval = y_cic.iloc[eval_idx].to_numpy()


## 3. Corrected CORAL implementation

Source data are aligned to the unlabeled target adaptation covariance. Target evaluation data stay in the target standardized coordinate system. The target evaluation labels are never used by CORAL.


In [ ]:

def symmetric_inv_sqrt(C, eps=1e-6):
    vals, vecs = np.linalg.eigh(C)
    vals = np.clip(vals, eps, None)
    return (vecs * (1.0 / np.sqrt(vals))) @ vecs.T


def symmetric_sqrt(C, eps=1e-6):
    vals, vecs = np.linalg.eigh(C)
    vals = np.clip(vals, eps, None)
    return (vecs * np.sqrt(vals)) @ vecs.T


def fit_coral(Xs, Xt):
    source_mean = Xs.mean(axis=0)
    target_mean = Xt.mean(axis=0)

    Zs = Xs - source_mean
    Zt = Xt - target_mean

    Cs = np.cov(Zs, rowvar=False)
    Ct = np.cov(Zt, rowvar=False)

    Cs_inv_sqrt = symmetric_inv_sqrt(Cs)
    Ct_sqrt = symmetric_sqrt(Ct)

    A = Cs_inv_sqrt @ Ct_sqrt

    return {
        "source_mean": source_mean,
        "target_mean": target_mean,
        "A": A,
        "Cs": Cs,
        "Ct": Ct,
    }


def apply_coral_source(X, coral):
    return (
        (X - coral["source_mean"]) @ coral["A"]
        + coral["target_mean"]
    )


def apply_target_standardization(X, target_mean, target_std):
    return (X - target_mean) / target_std


def covariance_distance(A, B):
    CA = np.cov(A, rowvar=False)
    CB = np.cov(B, rowvar=False)
    return float(np.linalg.norm(CA - CB, ord="fro"))


## 4. Train/evaluate controlled variants

All variants use the same source split, same target adaptation/evaluation rows, same Random Forest family, and source-derived thresholds. This makes the comparison meaningful.


In [ ]:

def select_threshold(y_true, proba):
    thresholds = np.linspace(0.05, 0.95, 91)
    scores = [
        f1_score(y_true, (proba >= t).astype(int))
        for t in thresholds
    ]
    best_idx = int(np.argmax(scores))
    return float(thresholds[best_idx])


def evaluate_binary(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, pred, labels=[0, 1]
    ).ravel()

    return {
        "f1": f1_score(y_true, pred, zero_division=0),
        "macro_f1": f1_score(
            y_true, pred, average="macro", zero_division=0
        ),
        "precision": precision_score(
            y_true, pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, pred, zero_division=0
        ),
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba),
        "fpr": fp / (fp + tn) if (fp + tn) else 0.0,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "predicted_attack_rate": float(pred.mean()),
        "threshold": float(threshold),
    }


RF_PARAMS = dict(
    n_estimators=250,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    max_features="sqrt",
    min_samples_leaf=2,
)


def run_variant(name, X_source, X_target, use_coral=False, balanced=False):
    Xs_fit, Xs_val, Xt_adapt, Xt_eval, imp = prepare_arrays(
        X_source,
        X_target,
        X_target
    )

    if use_coral:
        # Standardize using SOURCE statistics first.
        scaler = StandardScaler()
        Zs_fit = scaler.fit_transform(Xs_fit)
        Zs_val = scaler.transform(Xs_val)

        Zt_adapt = scaler.transform(Xt_adapt)
        Zt_eval = scaler.transform(Xt_eval)

        # CORAL uses only unlabeled target adaptation data.
        coral = fit_coral(Zs_fit, Zt_adapt)

        Xs_fit_model = apply_coral_source(Zs_fit, coral)
        Xs_val_model = apply_coral_source(Zs_val, coral)

        # Target data are represented in the target standardized
        # coordinate system. Do not multiply target data by A again.
        Xt_eval_model = Zt_eval

        cov_before = covariance_distance(Zs_fit, Zt_adapt)
        cov_after = covariance_distance(
            Xs_fit_model, Zt_adapt
        )
    else:
        Xs_fit_model = Xs_fit
        Xs_val_model = Xs_val
        Xt_eval_model = Xt_eval
        cov_before = np.nan
        cov_after = np.nan

    clf = RandomForestClassifier(
        **RF_PARAMS,
        class_weight="balanced" if balanced else None
    )

    clf.fit(Xs_fit_model, y_source_fit)

    source_val_proba = clf.predict_proba(
        Xs_val_model
    )[:, 1]

    threshold = select_threshold(
        y_source_val,
        source_val_proba
    )

    target_proba = clf.predict_proba(
        Xt_eval_model
    )[:, 1]

    metrics = evaluate_binary(
        y_target_eval,
        target_proba,
        threshold
    )

    result = {
        "model": name,
        "n_features": X_source.shape[1],
        "use_coral": use_coral,
        "balanced": balanced,
        "covariance_distance_before": cov_before,
        "covariance_distance_after": cov_after,
        **metrics,
    }

    return result, clf, target_proba


results = []
models = {}


# A. Current compact semantic representation
r, clf, proba = run_variant(
    "11-semantic",
    X_unsw_11,
    X_cic_11,
    use_coral=False,
    balanced=False
)
results.append(r)
models["11-semantic"] = (clf, proba)


# B. Current representation + CORAL + class weighting
r, clf, proba = run_variant(
    "11-semantic + CORAL + balanced",
    X_unsw_11,
    X_cic_11,
    use_coral=True,
    balanced=True
)
results.append(r)
models["11-semantic + CORAL + balanced"] = (clf, proba)


# C. Expanded representation
r, clf, proba = run_variant(
    "expanded-semantic",
    X_unsw_exp,
    X_cic_exp,
    use_coral=False,
    balanced=False
)
results.append(r)
models["expanded-semantic"] = (clf, proba)


# D. Expanded + CORAL
r, clf, proba = run_variant(
    "expanded-semantic + CORAL",
    X_unsw_exp,
    X_cic_exp,
    use_coral=True,
    balanced=False
)
results.append(r)
models["expanded-semantic + CORAL"] = (clf, proba)


# E. Expanded + CORAL + class weighting
r, clf, proba = run_variant(
    "expanded-semantic + CORAL + balanced",
    X_unsw_exp,
    X_cic_exp,
    use_coral=True,
    balanced=True
)
results.append(r)
models["expanded-semantic + CORAL + balanced"] = (clf, proba)


results_df = pd.DataFrame(results)

display(
    results_df[
        [
            "model",
            "n_features",
            "f1",
            "macro_f1",
            "pr_auc",
            "roc_auc",
            "precision",
            "recall",
            "fpr",
            "threshold",
            "predicted_attack_rate",
        ]
    ].sort_values("f1", ascending=False)
)

results_df.to_csv(
    TABLE_DIR / "01_expanded_semantic_comparison.csv",
    index=False
)


In [ ]:

# Identify the current best configuration.
best = results_df.sort_values(
    ["f1", "pr_auc", "roc_auc"],
    ascending=False
).iloc[0]

print("BEST CONFIGURATION")
print("------------------")
print("Model:", best["model"])
print("F1:", round(best["f1"], 4))
print("Macro-F1:", round(best["macro_f1"], 4))
print("PR-AUC:", round(best["pr_auc"], 4))
print("ROC-AUC:", round(best["roc_auc"], 4))
print("Recall:", round(best["recall"], 4))
print("FPR:", round(best["fpr"], 4))

# Compare against the current 0.339 project benchmark.
current_benchmark = 0.3385
improvement = best["f1"] - current_benchmark

print("\nCurrent Phase 12 benchmark F1:", current_benchmark)
print("New F1 minus Phase 12 benchmark:", round(improvement, 4))

results_df.to_csv(
    TABLE_DIR / "02_best_model_summary.csv",
    index=False
)


## 5. Interpretation rule

Do not automatically call the expanded representation better. It must beat the controlled 11-feature + CORAL + class-weighted baseline on the main metrics while keeping FPR reasonable.

If expanded semantic + CORAL + balanced improves F1/PR-AUC/ROC-AUC, it becomes the new candidate method. If it does not, the next step should be a more targeted unlabeled domain-adaptation method rather than adding more arbitrary features.


In [ ]:

# Save the experiment configuration for reproducibility.
config = {
    "random_state": RANDOM_STATE,
    "adaptation_n": int(len(adapt_idx)),
    "evaluation_n": int(len(eval_idx)),
    "compact_feature_count": int(X_unsw_11.shape[1]),
    "expanded_feature_count": int(X_unsw_exp.shape[1]),
    "rf_parameters": RF_PARAMS,
    "current_phase12_f1_benchmark": 0.3385,
    "leakage_rule": (
        "CICIDS labels used only for frozen evaluation; "
        "no target labels used for adaptation, thresholding, "
        "preprocessing, or feature selection."
    ),
}

with open(
    TABLE_DIR / "03_experiment_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, indent=2)

print("Saved outputs to:", TABLE_DIR)
